### 셀프 쿼리 ( Self-querying )

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_teddynote import logging

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [3]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

# 화장품 상품의 설명과 메타데이터 생성
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4.7},
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 4.5},
    ),
    Document(
        page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.",
        metadata={"year": 2023, "category": "클렌징", "user_rating": 4.8},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 4.6},
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4.4},
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.",
        metadata={"year": 2024, "category": "선케어", "user_rating": 4.9},
    ),
]

# 벡터 저장소 생성
vectorstore = Chroma.from_documents(
    docs, OpenAIEmbeddings(model="text-embedding-3-small")
)

- SelfQueryRetriever 
- retriever를 인스턴스화 할 수 있음. 이를 위해 문서가 지원하는 메타데이터 필드와, 
문서 내용에 대한 간단한 설명을 미리 제공해야 함

In [4]:
from langchain_classic.chains.query_constructor.base import AttributeInfo

# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The category of the cosmetic product. One of ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the cosmetic product was released",
        type="integer",
    ),
    AttributeInfo(
        name="user_rating",
        description="A user rating for the cosmetic product, ranging from 1 to 5",
        type="float",
    ),
]

In [5]:
# SelfQueryRetriever.from_llm() 메서드를 사용하여 retriever 객체를 생성

from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# SelfQueryRetriever 생성
retriever = SelfQueryRetriever.from_llm(
    llm= llm,
    vectorstore= vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info= metadata_field_info
)

ImportError: cannot import name 'DatabricksVectorSearch' from 'langchain_community.vectorstores' (d:\kingSJ\hanwha_0902\ex_0923\.venv\Lib\site-packages\langchain_community\vectorstores\__init__.py)

- Query 테스트
- 필터를 걸 수 있는 질의를 입력하여 검색을 수행

In [ ]:
# self-query 검색
retriever.invoke("평점이 4.8 이상인 제품을 추천해주세요.")

In [ ]:
# self-query
retriever.invoke("2023년에 출시된 상품을 추천해주세요.")

In [ ]:
# self-query 검색
retriever.invoke("카테고리가 선케어인 상품을 추천해주세요")

- 복합 필터를 사용하여 검색을 수행할 수 있음

In [ ]:
retriever.invoke("카테고리가 메이크업인 상품 중에서 평점이 4.5 이상인 상품을 추천해주세요.")

In [ ]:
retriever = SelfQueryRetriever.from_llm(
    llm = llm,
    vectorstore= vectorstore,
    document_contents= "Brief summary of a cosmetic product",
    metadata_field_info= metadata_field_info,
    enable_limit= True,
    search_kwargs = {"k" : 2}
)

In [ ]:
# 2023년도 출시된 상품은 3개가 있지만 k 값을 2로 지정하여 2개만 반환

retriever.invoke("2023년에 출시된 상품을 추천해주세요.")

In [ ]:
# kwargs를 지정하지 않고 query에서 1개, 2개 등의 숫자를 사용하여 결과 제한 가능

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    enable_limit=True,  # 검색 결과 제한 기능을 활성화합니다.
)

# Self-query 검색
retriever.invoke("2023년에 출시된 상품 1개를 추천해주세요")

In [ ]:
# Self-query 검색
retriever.invoke("2023년에 출시된 상품 2개를 추천해주세요")